# 🌬️ Air Quality ETL Pipeline — Deep Dive

A thorough walkthrough of the Madagascar air quality ETL: from Visual Crossing API to a queryable star schema in PostgreSQL.

---

## 🛠️ Pipeline Architecture

```
  Visual Crossing API
         ↓
  ┌──────────────────────────────────────────────┐
  │          AirQualityExtractor                 │
  │  ┌──────────┐  ┌──────────┐  ┌───────────┐ │
  │  │historical│  │ forecast │  │today_hrly │ │
  │  └────┬─────┘  └────┬─────┘  └─────┬─────┘ │
  └───────┼──────────────┼──────────────┼───────┘
          ↓              ↓              ↓
          └──────────────┬┴──────────────┘
                         ↓
  ┌──────────────────────────────────────────────┐
  │         DataFrameCleaner + DataValidator     │
  │         + DataAuditor (Quality Toolkit)      │
  └─────────────────────┬────────────────────────┘
                        ↓
  ┌──────────────────────────────────────────────┐
  │         Processed: daily_air_quality_combined │
  └─────────────────────┬────────────────────────┘
                        ↓
  ┌──────────────────────────────────────────────┐
  │  AirQualityTransformer  ──  dim_city        │
  │                        ──  dim_date         │
  │                        ──  fact_aqi         │
  │                        ──  fact_aqi_today   │
  └─────────────────────┬────────────────────────┘
                        ↓
  ┌──────────────────────────────────────────────┐
  │    CsvLoader (star_schema/*.csv)            │
  │    PostgresLoader (PostgreSQL)              │
  └──────────────────────────────────────────────┘
```

---
## 📌 Why This Architecture?

| Concern | Choice | What It Prevents |
|---------|--------|------------------|
| **Data corruption** | `DataValidator` range-checks every column before save | Garbage-in-garbage-out; catches API anomalies early |
| **Silent null creep** | `DataAuditor` logs every null %, duplicates, stats | Blind spots in data quality go unnoticed |
| **Broken pipelines** | `Settings.validate()` on startup | Runs with missing API keys or DB creds would fail midway |
| **Duplicate records** | `DataFrameCleaner.remove_duplicates()` + dedup on merge | Re-running the pipeline would balloon the data |
| **Mixed types in lists** | `_stringify_lists()` on extracted data | Pandas would coerce list cells into object arrays, breaking serialization |
| **API rate limits** | `@retry` decorator with exponential backoff | A 429 response kills the entire pipeline silently |
| **Schema evolution** | `save_star_schema()` overwrites via DELETE + append | Idempotent runs; re-runs don't create duplicates |
| **Referential integrity** | Foreign keys created in PostgreSQL | Orphan rows in fact tables would break BI reports |

---

## 1. ⚙️ Configuration Layer

### `config/settings.py`
Centralises all paths, env vars, and validators so nothing is hard-coded in business logic.

In [ ]:
from config.settings import Settings
from config.logging import setup_logging

setup_logging()
print(f"API Base URL: {Settings.VISUAL_CROSSING_BASE_URL}")
print(f"Raw data: {Settings.RAW_DIR}")
print(f"Clean data: {Settings.CLEAN_DIR}")
print(f"Star schema: {Settings.STAR_SCHEMA_DIR}")
print(f"Fact path: {Settings.AQI_FACT_PATH}")

### Why validate on startup?

```python
Settings.validate()
```
Checks every required env var **before** any network call. Without this, you might discover 5 minutes into extraction that `POSTGRES_PASSWORD` is missing. The pipeline fails fast — minutes saved.

In [ ]:
try:
    Settings.validate()
    Settings.ensure_directories()
    print("All settings valid. Directories ready.")
except ValueError as e:
    print(f"Validation failed: {e}")

---
## 2. 🧩 Quality Toolkit

Three classes every data frame passes through:

### 2a. DataValidator — Range Checking

**What it does:** Checks each numeric column against a known valid range.

**What it prevents:** If the API returns `pm2.5: 9999` (a sensor error), the validator flags it before the bad value enters the star schema. BI dashboards won't show a 9999 µg/m³ spike.

**Ranges defined:**

In [ ]:
from src.transform.quality.data_validator import DataValidator

for col, (lo, hi) in DataValidator.RANGES.items():
    print(f"  {col:20s}  [{lo:>6}, {hi:>6}]")
print(f"\nTotal metrics tracked: {len(DataValidator.RANGES)}")

**Live demo — validation in action:**

In [ ]:
import pandas as pd
import logging

# Good data: all within range
good = pd.DataFrame({"pm2.5": [10, 25, 35], "pm10": [20, 40, 60]})
print("=== Valid data ===")
DataValidator.validate(good, "Good Example")

print()

# Bad data: sensor spike
bad = pd.DataFrame({"pm2.5": [10, 999, 35], "o3": [30, 40, 9999]})
print("=== Corrupted data (should warn) ===")
DataValidator.validate(bad, "Bad Example")

The warning tells exactly which column, how many values are out of range, and what the actual min/max are — no guessing.

### 2b. DataFrameCleaner — Sanitisation

**What it does:** Six composable static methods that each fix one class of problem.

**What it prevents:**
- Empty strings (`""`, `"  "`) that look like valid data but break joins → `normalize_empty_strings()` converts them to `NaN`
- Duplicate rows that inflate averages → `remove_duplicates()` drops them
- Nulls in numeric columns break calculations → `fill_numeric_nulls()` fills with 0
- Nulls in categorical columns break GROUP BY → `fill_categorical_nulls()` fills with `"unknown"`
- Columns that are entirely null waste space → `drop_all_null_columns()` removes them

In [ ]:
from src.transform.quality.dataframe_cleaner import DataFrameCleaner

# Simulate messy raw data
messy = pd.DataFrame({
    "pm2.5": [10.0, None, 20.0, None],
    "pm10": [None, 15.0, 25.0, None],
    "city_name": ["Tana", "", "  ", "Fianar"],
    "useless": [None, None, None, None],
})

print("=== BEFORE cleaning ===")
print(messy)
print(f"\nNull counts:\n{messy.isnull().sum()}")

clean = DataFrameCleaner.clean_air_quality_data(messy)

print("\n=== AFTER cleaning ===")
print(clean)
print(f"\nNull counts:\n{clean.isnull().sum()}")

Notice:
- `""` and `"  "` became `"unknown"` (normalised to NaN, then filled)
- `pm2.5` and `pm10` nulls became 0
- `useless` column dropped entirely

### 2c. DataAuditor — Observability

**What it does:** Runs a full health check on any DataFrame — shape, nulls, duplicates, numeric stats, categorical distribution.

**What it prevents:** Data degradation that happens gradually. If one city's extractor starts returning all nulls, the auditor catches it the same day.

**Why not just trust the validator?** The validator checks values are in range; the auditor checks **structure** — are we getting the right number of rows? Are there sudden spikes in nulls? Are we losing columns?

In [ ]:
from src.transform.quality.data_auditor import DataAuditor

auditor = DataAuditor()

sample = pd.DataFrame({
    "pm2.5": [10.0, None, 35.0, 20.0, 15.0],
    "pm10": [20.0, 30.0, None, 40.0, 25.0],
    "city_name": ["Tana", "Tana", "Tana", "Fianar", "Fianar"],
})

report = auditor.audit_dataframe(sample, "Demo Data")

# Peek at the structured report
print(f"Rows: {report['basic_info']['row_count']}")
print(f"Cols: {report['basic_info']['column_count']}")
print(f"Total nulls: {report['null_analysis']['total_nulls']}")
print(f"Duplicates: {report['duplicate_analysis']['duplicate_count']}")
print(f"Numeric columns tracked: {list(report['numeric_statistics'].keys())}")

---
## 3. 📂 Data Flow — End to End

### Step 1: Extract
Each city goes through three extractions:

In [ ]:
# Pseudocode — WeatherExtractor is built by Dev 1 & 2
# extractor = AirQualityExtractor(api_key, base_url)
# historical = extractor.extract_historical("Antananarivo")   # last 5 days
# forecast   = extractor.extract_forecast("Antananarivo")     # next 5 days
# hourly     = extractor.extract_today_hourly("Antananarivo") # today's 24 hours
print("Extraction produces 3 DataFrames per city:")
print("  historical  → data/raw/historical/{city}_historical.csv")
print("  forecast    → data/raw/forecast/{city}_forecast.csv")
print("  today_hrly  → data/raw/today_hourly/{city}_today_hourly.csv")

**Why 3 separate folders?**

Each extract type has different columns and update frequency:
- Historical: daily snapshot, never changes
- Forecast: daily snapshot, changes every API call
- Hourly: 24 rows per city per day, high volume

Mixing them would force every consumer to filter by type.

### Step 2: Clean

Every raw DataFrame passes through `DataFrameCleaner.clean_air_quality_data()`:

In [ ]:
cleanup_pipeline = [
    "1. normalize_empty_strings()   —  '' / '   ' → NaN",
    "2. remove_duplicates()         —  drop exact row copies",
    "3. fill_numeric_nulls()        —  NaN → 0",
    "4. fill_categorical_nulls()    —  NaN → 'unknown'",
]
for step in cleanup_pipeline:
    print(step)

### Step 3: Validate + Audit

Before anything enters the star schema, we run both:

In [ ]:
# After combining historical + forecast:
print("Fact table assembly checks:")
print("  1. DataAuditor.audit_dataframe(fact_aqi, 'Fact AQI')")
print("  2. DataValidator.validate(fact_aqi, 'Fact AQI')")
print()
print("If either step logs warnings, the pipeline still completes")
print("(but the warnings appear in production logs for review).")

### Step 4: Star Schema

The processed daily data is transformed into a star schema:

In [ ]:
print("""
   dim_city           dim_date
   ─────────          ─────────
   city_key PK        date_key PK
   city_name          full_date
   region             year
   latitude           month
   longitude          day
   population         day_of_week
                      quarter
        \\            //
         \\          //
        fact_aqi / fact_aqi_today
       ─────────────────────────
       city_key FK → dim_city
       date_key FK → dim_date
       hour (only fact_aqi_today)
       pm2.5, pm10, o3, no2, so2, co
       aqius, pm1
""")

### Step 5: Load

Two destinations, same data:

In [ ]:
print("CSV (data/star_schema/)")
for f in ["dim_date.csv", "dim_city.csv", "fact_aqi.csv", "fact_aqi_today.csv"]:
    print(f"  └── {f}")

print("\nPostgreSQL (air_quality schema)")
print("  Tables: dim_city, dim_date, fact_aqi, fact_aqi_today")
print("  Strategy: DELETE + append (idempotent)")
print("  Constraints: PKs, UNIQUE, FKs with CASCADE")

**Why DELETE + append, not REPLACE?**

`df.to_sql(if_exists='replace')` drops and recreates the table — this kills foreign keys, indexes, and any data other processes wrote. DELETE + append preserves schema and constraints.

**Why unique constraints on facts?** Re-running the pipeline shouldn't double rows. `UNIQUE (city_key, date_key)` on `fact_aqi` guarantees idempotency.

---
## 4. ⚠️ Failure Scenarios & Prevention

| Scenario | What Breaks | How We Prevent It |
|----------|-------------|-------------------|
| API key invalid | All extracts fail | `Settings.validate()` fails fast at startup |
| API rate limit (429) | Single city's extract fails | `@retry` retries 3× with backoff |
| API returns `pm2.5: -5` | Negative values in BI | `DataValidator.RANGES[pm2.5] = (0, 500)` flags it |
| Empty string in city name | Broken joins in Power BI | `normalize_empty_strings()` → `NaN` → `fillna('unknown')` |
| Pipeline killed mid-run | Duplicate rows on re-run | `remove_duplicates()` + `UNIQUE` constraint |
| New column added by API | Unrecognised column silently passes | `DataAuditor` logs every column found vs expected |
| PostgreSQL down at 6 AM | Pipeline crashes after all extracts | `save_postgres` is the **last** step; CSV files are already safe |
| Missing city CSV | No cities to process | `CityExtractor` raises `FileNotFoundError` immediately |

---

## 5. 🧩 Running the Pipeline

### One-shot execution
```bash
python main.py
```

### Scheduled via Airflow
The DAG `air_quality_madagascar_etl` runs daily at 06:00:

```
validate_settings
       ↓
extract_{historical,forecast,hourly}_{city}  (parallel per city)
       ↓
prepare_dimensions
       ↓
transform_facts
       ↓
save_postgres
```

### What you get
```
data/
  raw/{historical, forecast, today_hourly}/{city}_{type}.csv
  clean/daily_air_quality_combined.csv
  star_schema/{dim_date,dim_city,fact_aqi,fact_aqi_today}.csv
```

PostgreSQL `air_quality` schema with 4 tables, foreign keys, and unique constraints.

---
## 6. 📊 Summary Decision Table

| Decision | Rationale |
|----------|-----------|
| Visual Crossing API | Free tier provides PM2.5, PM10, NO2, O3, CO, SO2, AQI for any location |
| 5-day windows | Free tier limits; historical + forecast = 10 days of daily data per run |
| Raw CSV per city per type | Append-only on re-run; allows per-city debugging |
| Daily combined CSV | Single source of truth for Power BI connect |
| Star schema | BI-optimised; dimensions shared across facts |
| `fact_aqi_today` hourly | Enables hour-of-day analysis (rush hour vs night) |
| DELETE + append | Preserves schema, indexes, FKs; idempotent |
| Range validation | Catches sensor errors before they poison aggregates |
| Audit logging | Structural health check (null creep, column drift) |
| `0` for numeric nulls | Keeps averages meaningful; `mean()` ignores NaN |
| `"unknown"` for cat nulls | Preserves GROUP BY without dropping rows |
| `@retry` decorator | Free tier throttles at ~1000 calls/day; retries absorb bursts |
| `Settings.validate()` first | Fail fast before any network call |

---
*End of pipeline deep dive.*